# PARC2026 — 69b OpenVLA selected-subset RLDS bridge smoke

D10で確定した `V2_SQRT_BALANCED_RAW` の **同じepisode pool** を OpenVLA-OFTへ渡すための LeRobot → RLDS/TFDS bridge を、小さい8 episodeだけで検証します。

このNotebookは **full RLDSを作りません**。先に8 episodeを変換して、schema・画像/State/Action・provenanceと実容量を確認し、10,758 episodeを丸ごとmaterializeした場合のDrive容量を推定します。

- CPU / L4 / A100で実行可（GPU学習なし）
- sourceはDrive上の `lerobot/libero_plus` v3 / 20Hz
- front → OpenVLA `image`, wrist → `wrist_image`
- state8 / action7は変換時に値を書き換えない
- no-op除去もしない（公平比較のsame episode poolを優先）
- full manifest全部を変換しない限り `conversion_contract.json` は作らない
- 30秒heartbeatで進捗を表示


In [ ]:
# 0/5 Drive + D10 + repo + heartbeat
import os, json, shutil, subprocess, sys, threading, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
D10 = DRIVE/'pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json'
assert D10.exists(), D10
decision=json.loads(D10.read_text())
assert decision.get('status')=='DECIDED', decision
assert decision.get('selected_variant')=='V2_SQRT_BALANCED_RAW', decision
print('D10:', decision['selected_variant'], 'seed=', decision['eval_seed'])

ROOT=Path('/content/parc2026'); REPO=ROOT/'py_AI'; ROOT.mkdir(parents=True,exist_ok=True)
if not (REPO/'.git').exists(): subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force','origin/main'],check=True)
print('repo:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

SRC=DRIVE/'datasets/lerobot_libero_plus_v3_train'
assert (SRC/'meta/info.json').exists(), SRC
OUT=DRIVE/'openvla-rlds-selected-v1'; OUT.mkdir(parents=True,exist_ok=True)

def _device_line():
    try:
        return subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True).strip()
    except Exception:
        return 'CPU/no nvidia-smi'

def run_hb(cmd, *, label, cwd=None, env=None):
    print('>>>', ' '.join(map(str,cmd)), flush=True)
    start=time.time(); p=subprocess.Popen(cmd,cwd=cwd,env=env)
    while p.poll() is None:
        free=shutil.disk_usage('/content').free/1024**3
        print(f'[heartbeat] {label} elapsed={(time.time()-start)/60:.1f}m device={_device_line()} free={free:.1f}GiB',flush=True)
        time.sleep(30)
    if p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd)
    print(f'[done] {label} elapsed={(time.time()-start)/60:.1f}m rc=0',flush=True)

print('source:',SRC)
print('local free:',f'{shutil.disk_usage("/content").free/1024**3:.1f} GiB')
print('=== PREFLIGHT: PASS ===')


In [ ]:
# 1/5 Locate V2 manifest; regenerate only data-selection artifacts if Drive copy is absent
import json, subprocess, sys
from pathlib import Path

expected_name='V2_SQRT_BALANCED_RAW.json'
likely=[
 DRIVE/'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware'/expected_name,
 DRIVE/'pi05-ablation-group-aware-v2/outputs/dataset_ablation_manifests_v2_group_aware'/expected_name,
 DRIVE/'dataset_ablation_manifests_v2_group_aware'/expected_name,
]
MANIFEST=None
for p in likely:
    if p.exists():
        d=json.loads(p.read_text())
        if d.get('schema_version')==2 and d.get('group_aware') is True and d.get('variant')=='V2_SQRT_BALANCED_RAW': MANIFEST=p; break

if MANIFEST is None:
    print('Drive manifest not found -> regenerate deterministic group-aware artifacts from parquet only.')
    subprocess.run([sys.executable,'-m','pip','install','-q','pandas>=2','pyarrow>=16'],check=True)
    work=ROOT/'outputs/rlds_bridge_manifest_rebuild'; work.mkdir(parents=True,exist_ok=True)
    static=work/'static_quality_v1'; legacy=work/'legacy_v1'; leak1=work/'leak_v1'; mroot=work/'v2'; leak2=work/'leak_v2'
    run_hb([sys.executable,'-u',str(REPO/'tools/data/static_quality_analyzer.py'),'--root',str(SRC),'--out',str(static),'--smooth-window','5','--robust-z-threshold','5'],label='static-quality')
    run_hb([sys.executable,'-u',str(REPO/'tools/data/build_dataset_ablation_manifests.py'),'--metrics-csv',str(static/'episode_quality_metrics.csv'),'--out',str(legacy),'--dataset-id','lerobot/libero_plus','--dataset-revision','f3f49f426d75030177b18778374005bc12ccd588','--seed','20260830','--eval-per-task','2'],label='legacy-manifest')
    run_hb([sys.executable,'-u',str(REPO/'tools/data/check_trajectory_group_leakage.py'),'--root',str(SRC),'--manifests-dir',str(legacy),'--out',str(leak1),'--metrics-csv',str(static/'episode_quality_metrics.csv'),'--round-decimals','6'],label='trajectory-groups')
    run_hb([sys.executable,'-u',str(REPO/'tools/data/build_group_aware_ablation_manifests.py'),'--metrics-csv',str(static/'episode_quality_metrics.csv'),'--trajectory-groups-csv',str(leak1/'trajectory_group_members.csv'),'--out',str(mroot),'--dataset-id','lerobot/libero_plus','--dataset-revision','f3f49f426d75030177b18778374005bc12ccd588','--seed','20260830','--eval-per-task','2'],label='group-aware-manifest')
    run_hb([sys.executable,'-u',str(REPO/'tools/data/check_trajectory_group_leakage.py'),'--root',str(SRC),'--manifests-dir',str(mroot),'--out',str(leak2),'--metrics-csv',str(static/'episode_quality_metrics.csv'),'--round-decimals','6','--fail-on-exact-leakage'],label='group-aware-leak-gate')
    MANIFEST=mroot/expected_name

m=json.loads(MANIFEST.read_text())
assert m['schema_version']==2 and m['group_aware'] is True
assert m['variant']=='V2_SQRT_BALANCED_RAW'
assert m['dataset_id']=='lerobot/libero_plus'
assert m['dataset_revision']=='f3f49f426d75030177b18778374005bc12ccd588'
assert m['summary']['episode_count']==10758, m['summary']
print('manifest:',MANIFEST)
print('episodes:',m['summary']['episode_count'],'frames:',m['summary']['frame_count'])
print('episode_ids_sha256:',m['episode_ids_sha256'])
print('=== V2 MANIFEST GATE: PASS ===')


In [ ]:
# 2/5 Isolated Python 3.10 conversion environment (no model weights)
import subprocess, sys
from pathlib import Path

if shutil.which('uv') is None: subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','python','install','3.10'],check=True)
VENV=ROOT/'venv-openvla-rlds'; PY=VENV/'bin/python'
if not PY.exists(): subprocess.run(['uv','venv','--python','3.10',str(VENV)],check=True)
deps=['numpy<2','pandas>=2,<3','pyarrow>=16','tensorflow-cpu==2.17.1','tensorflow-datasets==4.9.6','av>=12,<15']
run_hb(['uv','pip','install','--python',str(PY),*deps],label='rlds-conversion-deps')
subprocess.run([str(PY),'-c','import tensorflow as tf, tensorflow_datasets as tfds, av, pyarrow; print("tf",tf.__version__,"tfds",tfds.__version__,"av",av.__version__)'],check=True)
print('=== CONVERSION ENV: PASS ===')


In [ ]:
# 3/5 Convert only 8 exact V2 episodes and validate prepared TFDS
SMOKE_ROOT=ROOT/'openvla-rlds-smoke-v1'
if SMOKE_ROOT.exists(): shutil.rmtree(SMOKE_ROOT)
REPORT=OUT/'bridge_smoke_report.json'
cmd=[str(PY),'-u',str(REPO/'tools/data/convert_lerobot_manifest_to_openvla_rlds.py'),'--lerobot-root',str(SRC),'--manifest',str(MANIFEST),'--out-root',str(SMOKE_ROOT),'--report',str(REPORT),'--dataset-repo-id','lerobot/libero_plus','--dataset-revision','f3f49f426d75030177b18778374005bc12ccd588','--max-episodes','8','--overwrite']
run_hb(cmd,label='LeRobot->RLDS 8ep smoke')
r=json.loads(REPORT.read_text())
assert r['status']=='SMOKE_PASS',r
assert r['selected_dataset_variant']=='V2_SQRT_BALANCED_RAW'
assert r['tfds_builder_from_directory']=='PASS'
assert r['converted_episode_count']==8
assert not (OUT/'conversion_contract.json').exists(), 'smoke must not create full conversion contract'
print(json.dumps(r,indent=2))
print('=== RLDS BRIDGE SMOKE: PASS ===')


In [ ]:
# 4/5 Capacity decision — do not full-convert automatically
import json
THRESHOLD_GIB=35.0
r=json.loads(REPORT.read_text())
projected=float(r['projected_full_gib'])
decision_code='MATERIALIZE_CANDIDATE' if projected <= THRESHOLD_GIB else 'STREAMING_BRIDGE_RECOMMENDED'
capacity={
 'schema_version':1,
 'status':'PASS',
 'selected_dataset_variant':'V2_SQRT_BALANCED_RAW',
 'smoke_episode_count':r['converted_episode_count'],
 'smoke_frames':r['converted_frames'],
 'bytes_per_frame':r['bytes_per_frame'],
 'projected_full_gib':projected,
 'materialize_threshold_gib':THRESHOLD_GIB,
 'decision':decision_code,
 'next_step':('full exact-manifest conversion may be considered' if decision_code=='MATERIALIZE_CANDIDATE' else 'implement streaming LeRobot->OpenVLA adapter; do not duplicate the full video dataset into TFRecords'),
 'm3_budget_file':'experiments/plans/model_benchmark_budget_v1.json'
}
(OUT/'bridge_capacity_decision.json').write_text(json.dumps(capacity,indent=2)+'\n')
print(json.dumps(capacity,indent=2))
print('=== 69b COMPLETE ===')
print('Paste bridge_smoke_report + bridge_capacity_decision back to ChatGPT before running notebook 70.')
